# Silver Layer — CDF-Driven Incremental MERGE

## Architecture
Silver reads **only the rows that changed in Bronze** using Delta's **Change Data Feed (CDF)**,
then applies a `MERGE INTO` upsert — touching the minimum number of records per run.

## Design Decisions
| Feature | Decision |
|---|---|
| Incremental source | CDF (`table_changes`) — reads inserts/updates/deletes since last committed version |
| Write pattern | `MERGE INTO` keyed on `primary_key` from config |
| Schema evolution | `ALTER TABLE ... ADD COLUMNS` auto-applied before MERGE |
| Deduplication | De-dup on PK within each CDF micro-batch before merge |
| Audit column | `silver_load_timestamp` tracks when the Silver row was last written |
| CDF on Silver | Enabled — allows Gold to do the same CDF-incremental pattern |
| Version tracking | Last processed Bronze version stored in `framework.silver_watermark` |

## CDF Version Watermark
A `silver_watermark` table tracks the last Bronze Delta version processed per source.
On first run: reads from version 0. On subsequent runs: reads only new versions. This
makes Silver completely idempotent and safe to re-run.

In [ ]:
%run ./fw_0.config

In [ ]:
# ── Step 1: Provision Silver schema & watermark table ─────────────────────────

from pyspark.sql import functions as F, Row
from pyspark.sql.utils import AnalysisException
from datetime import datetime

spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}
    MANAGED LOCATION '{SILVER_PATH}'
    COMMENT 'Cleansed, deduplicated Silver layer — CDF-driven incremental MERGE'
""")

WATERMARK_TABLE = fq(FRAMEWORK_SCHEMA, "silver_watermark")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {WATERMARK_TABLE} (
        source_name           STRING    NOT NULL COMMENT 'Matches pipeline_config.source_name',
        last_bronze_version   BIGINT    NOT NULL COMMENT 'Last Bronze Delta version read by Silver',
        updated_at            TIMESTAMP NOT NULL
    )
    USING DELTA
    TBLPROPERTIES ('quality' = 'framework')
    COMMENT 'CDF watermark — tracks last Bronze version processed per source'
""")
print(f"Watermark table ready: {WATERMARK_TABLE}")

In [ ]:
# ── Step 2: Silver processing functions ───────────────────────────────────────

def get_watermark(source_name: str) -> int:
    """Return the last Bronze version processed for this source. 0 on first run."""
    rows = (
        spark.table(WATERMARK_TABLE)
        .filter(F.col("source_name") == source_name)
        .collect()
    )
    return int(rows[0].last_bronze_version) if rows else 0


def update_watermark(source_name: str, version: int):
    """Upsert the watermark for this source to the latest Bronze version processed."""
    spark.createDataFrame([
        Row(source_name=source_name, last_bronze_version=version, updated_at=datetime.utcnow())
    ]).createOrReplaceTempView("wm_update")
    spark.sql(f"""
        MERGE INTO {WATERMARK_TABLE} AS tgt
        USING wm_update AS src
        ON tgt.source_name = src.source_name
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)


def get_latest_bronze_version(bronze_table: str) -> int:
    """Return the current Delta table version of the Bronze table."""
    detail = spark.sql(f"DESCRIBE HISTORY {bronze_table} LIMIT 1").collect()
    return int(detail[0].version) if detail else 0


def evolve_silver_schema(silver_table: str, source_df):
    """
    Add any columns present in source_df but missing from the Silver table.
    Prevents MERGE from failing on schema mismatch after Bronze schema evolution.
    """
    if not table_exists(SILVER_SCHEMA, silver_table.split(".")[-1]):
        return  # Table doesn't exist yet; will be created by first MERGE
    existing_cols = {f.name.lower() for f in spark.table(silver_table).schema.fields}
    for field in source_df.schema.fields:
        if field.name.lower() not in existing_cols:
            spark.sql(f"""
                ALTER TABLE {silver_table}
                ADD COLUMN {field.name} {field.dataType.simpleString()}
            """)
            print(f"         Schema evolved: added column '{field.name}' to {silver_table}")


def build_merge_condition(pk_cols: list, alias_tgt: str = "tgt", alias_src: str = "src") -> str:
    """Build the ON clause for MERGE from a list of primary key column names."""
    return " AND ".join(f"{alias_tgt}.{pk} = {alias_src}.{pk}" for pk in pk_cols)


print("Silver utility functions loaded.")

In [ ]:
# ── Step 3: Core Silver MERGE function ───────────────────────────────────────

def process_silver(cfg) -> dict:
    """
    Incrementally merge Bronze CDF changes into Silver for one pipeline_config row.
    Returns a result dict with status and rows_affected.
    """
    source_name   = cfg.source_name
    bronze_table  = fq(BRONZE_SCHEMA, cfg.bronze_table)
    silver_table  = fq(SILVER_SCHEMA, cfg.silver_table)
    pk_cols       = [pk.strip() for pk in cfg.primary_key.split(",")]
    partition_col = cfg.partition_col or "ingestion_date"
    zorder_cols   = [c.strip() for c in (cfg.zorder_cols or "").split(",") if c.strip()]
    load_type     = cfg.silver_load_type.lower()

    print(f"\n[SILVER] Starting: {source_name}  →  {silver_table}")
    print(f"         Bronze   : {bronze_table}")
    print(f"         Load type: {load_type}  |  PK: {pk_cols}")

    # ── Read CDF changes since last watermark ──────────────────────────────
    from_version    = get_watermark(source_name)
    latest_version  = get_latest_bronze_version(bronze_table)

    if from_version >= latest_version:
        print(f"         [SKIP] No new Bronze versions (watermark={from_version}, latest={latest_version})")
        return {"status": "SKIPPED", "rows": 0}

    cdf_df = (
        spark.read
        .format("delta")
        .option("readChangeFeed",      "true")
        .option("startingVersion",     from_version + 1)
        .table(bronze_table)
        # Keep only inserts and updates; deletes propagate as NULL rows from Bronze
        .filter(F.col("_change_type").isin("insert", "update_postimage"))
        .drop("_change_type", "_commit_version", "_commit_timestamp")
    )

    # Deduplicate within the CDF batch on primary key — keep last version per key
    from pyspark.sql.window import Window
    window_spec = Window.partitionBy(*pk_cols).orderBy(F.col("load_timestamp").desc())
    cdf_deduped = (
        cdf_df
        .withColumn("_row_num", F.row_number().over(window_spec))
        .filter(F.col("_row_num") == 1)
        .drop("_row_num")
        .withColumn("silver_load_timestamp", F.current_timestamp())
    )

    if cdf_deduped.isEmpty():
        print(f"         [SKIP] CDF batch is empty after deduplication.")
        update_watermark(source_name, latest_version)
        return {"status": "SKIPPED", "rows": 0}

    # ── Schema evolution ───────────────────────────────────────────────────
    evolve_silver_schema(silver_table, cdf_deduped)

    # ── MERGE INTO Silver ──────────────────────────────────────────────────
    merge_condition = build_merge_condition(pk_cols)
    cdf_deduped.createOrReplaceTempView(f"silver_src_{source_name}")

    if load_type == "merge":
        spark.sql(f"""
            MERGE INTO {silver_table} AS tgt
            USING silver_src_{source_name} AS src
            ON {merge_condition}
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
    else:  # append
        cdf_deduped.write.format("delta").mode("append") \
            .option("mergeSchema", "true") \
            .saveAsTable(silver_table)

    # ── Set Silver TBLPROPERTIES ───────────────────────────────────────────
    spark.sql(f"""
        ALTER TABLE {silver_table}
        SET TBLPROPERTIES (
            'delta.enableChangeDataFeed' = 'true',
            'quality'                    = 'silver'
        )
    """)

    # ── OPTIMIZE + ZORDER ─────────────────────────────────────────────────
    if zorder_cols:
        spark.sql(f"OPTIMIZE {silver_table} ZORDER BY ({', '.join(zorder_cols)})")
    else:
        spark.sql(f"OPTIMIZE {silver_table}")

    # ── Advance watermark ─────────────────────────────────────────────────
    update_watermark(source_name, latest_version)

    row_count = spark.table(silver_table).count()
    print(f"[SILVER] Done: {silver_table}  →  {row_count:,} total rows  (watermark → {latest_version})")
    return {"status": "SUCCESS", "rows": row_count}


print("Silver processing function loaded.")

In [ ]:
# ── Step 4: Process all active Silver configs ──────────────────────────────────

configs = get_pipeline_configs(filter_active=True)
results = []

print(f"Active configs: {len(configs)}")

for cfg in configs:
    try:
        result = process_silver(cfg)
        log_pipeline_run(cfg.config_id, "silver", result["status"], result["rows"])
        results.append({"source": cfg.source_name, **result})

    except Exception as e:
        error_msg = str(e)
        print(f"[SILVER] FAILED: {cfg.source_name}  →  {error_msg}")
        log_pipeline_run(cfg.config_id, "silver", "FAILURE", error_msg=error_msg)
        results.append({"source": cfg.source_name, "status": "FAILURE", "error": error_msg})

# ── Summary ──────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("SILVER MERGE SUMMARY")
print("="*60)
for r in results:
    rows = r.get('rows', 'N/A')
    print(f"  {r['source']:20s}  {r.get('status','?'):10s}  {str(rows):>12s} rows")

In [ ]:
# ── Step 5: Data Quality Gate ────────────────────────────────────────────────

failures = [r for r in results if r.get("status") == "FAILURE"]
if failures:
    failed_sources = ", ".join(r["source"] for r in failures)
    raise RuntimeError(
        f"[DQ FAIL] Silver processing failed for: {failed_sources}. "
        "Check pipeline_run_log for details."
    )

print("[DQ PASS] All Silver sources processed successfully.")